# IEEE 33-Bus Constraint-First Dataset Generator (HDF5)

This notebook creates a physically realistic IEEE 33-bus distribution-system dataset for Graph Transformer research. The dataset is designed for voltage-stability prediction, loss estimation, line-loading prediction, switching behavior modeling, and topology-aware learning.

## Constraint ranges enforced by this notebook

| Constraint | Normal target | Mild stress target | Extreme target | Hard reject |
|---|---:|---:|---:|---:|
| Minimum voltage | `Vmin >= 0.90 pu` | `0.85 <= Vmin < 0.90 pu` | `0.80 <= Vmin < 0.85 pu` | `Vmin < 0.80 pu` |
| Line loading | `< 80%` | `80-95%` | `95-100%` | `> 100%` |
| Total losses | `50-300 kW` | `300-500 kW` | `500-600 kW` | `> 600 kW` |
| Topology | radial tree | radial tree | radial tree | disconnected, looped, non-tree |
| Switch deviation from base | `0-2 line-status changes` | `2-4 line-status changes` | `2-4 line-status changes` | `> 4 line-status changes` |
| Load scaling | `0.90-1.05` | `1.05-1.20` | `1.20-1.30` | outside calibrated bounds |

## Final dataset target mix

- Normal: `60-70%`
- Mild stress: `20-30%`
- Extreme stress: `5-10%`

## Design principle

The pipeline uses **constraint-first acceptance sampling**:

1. Precompute a limited set of realistic radial switching states (`30-80` topologies total).
2. Sample calibrated bounded loads.
3. Select only from valid precomputed topologies.
4. Apply early physical-risk checks before power flow.
5. Run `pandapower` only on plausible candidates.
6. Store only samples that pass all hard constraints.

This dataset is **not** designed to maximize topology diversity artificially. Topology diversity is intentionally limited by IEEE-33 operating realism; load variation is the primary learning signal.

**HDF5 output**: `data/ieee33_dataset.h5`


In [ ]:
import json

import pandapower as pp
import pandapower.networks as pn
import networkx as nx
import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
import h5py
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

GLOBAL_SEED = 20260622
N_SAMPLES = 1000
BATCH_SIZE = 64
N_JOBS = -1
H5_PATH = "data/ieee33_dataset.h5"
COMPRESSION = "gzip"
COMPRESSION_OPTS = 4

VOLTAGE_NORMAL_MIN = 0.90
VOLTAGE_MILD_MIN = 0.85
VOLTAGE_HARD_MIN = 0.80
VOLTAGE_MAX = 1.05
LOADING_NORMAL_MAX = 80.0
LOADING_MILD_MAX = 95.0
LOADING_HARD_MAX = 100.0
LOSS_NORMAL_MAX_KW = 300.0
LOSS_MILD_MAX_KW = 500.0
LOSS_HARD_MAX_KW = 600.0
MIN_REALISTIC_LOSS_KW = 20.0

TARGET_CATEGORY_POLICY = {"normal": 0.65, "mild_stress": 0.27, "extreme_stress": 0.08}
LOAD_SPECS = {
    "normal": {"mean": 0.98, "std": 0.030, "clip": (0.90, 1.05), "total_bounds": (0.93, 1.03)},
    "mild_stress": {"mean": 1.11, "std": 0.035, "clip": (1.05, 1.20), "total_bounds": (1.06, 1.16)},
    "extreme_stress": {"mean": 1.23, "std": 0.030, "clip": (1.20, 1.30), "total_bounds": (1.18, 1.26)},
}

MIN_PRECOMPUTED_TOPOLOGIES = 30
TARGET_PRECOMPUTED_TOPOLOGIES = 60
MAX_PRECOMPUTED_TOPOLOGIES = 80
MAX_SWITCH_DEVIATION = 4
MAX_BRANCH_EXCHANGES = 2
MAX_GENERATION_ATTEMPT_MULTIPLIER = 8

NODE_FEATURE_NAMES = ["p_load_mw", "q_load_mvar", "feeder_depth", "distance_to_substation", "bus_type", "initial_voltage_pu"]
EDGE_FEATURE_NAMES = ["r_ohm", "x_ohm", "switch_status", "impedance_magnitude"]
GLOBAL_FEATURE_NAMES = ["total_load_mw", "num_buses", "num_lines", "active_lines", "topology_deviation"]
LABEL_NAMES = ["total_loss_kw", "min_voltage_pu", "max_line_loading_percent", "voltage_violation_count", "overload_count", "constraint_violation_count"]

np.random.seed(GLOBAL_SEED)
plt.rcParams["figure.figsize"] = (9, 5)
print(f"Configuration ready | samples={N_SAMPLES:,} | seed={GLOBAL_SEED}")


## STEP 1 — Load IEEE 33 Bus System

`load_ieee33()` uses `pandapower.networks.case33bw()`. The notebook also prepares the standard IEEE-33 normally-open tie branches and normalizes placeholder line-current ratings to realistic study values for loading labels.


In [ ]:
STANDARD_TIE_BRANCH_DATA = [
    {"from_bus": 20, "to_bus": 7, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 8, "to_bus": 14, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 11, "to_bus": 21, "r_ohm": 2.0, "x_ohm": 2.0},
    {"from_bus": 17, "to_bus": 32, "r_ohm": 0.5, "x_ohm": 0.5},
    {"from_bus": 24, "to_bus": 28, "r_ohm": 0.5, "x_ohm": 0.5},
]

BASE_NET_JSON = None
BASE_LINE_STATUS = None
BASE_GRAPH = None
BASE_TOTAL_LOAD_MW = None
BASE_POWERFLOW_RESULT = None
VALID_TOPOLOGIES = None


def _ensure_line_columns(net):
    if "in_service" not in net.line.columns:
        net.line["in_service"] = True
    net.line["in_service"] = net.line["in_service"].fillna(True).astype(bool)
    if "is_tie" not in net.line.columns:
        net.line["is_tie"] = False
    net.line["is_tie"] = net.line["is_tie"].fillna(False).astype(bool)


def _line_mask_for_endpoints(net, from_bus, to_bus):
    from_values = net.line["from_bus"].astype(int).to_numpy()
    to_values = net.line["to_bus"].astype(int).to_numpy()
    return ((from_values == int(from_bus)) & (to_values == int(to_bus))) | ((from_values == int(to_bus)) & (to_values == int(from_bus)))


def _median_or_default(series, default_value):
    if series is None or len(series) == 0:
        return float(default_value)
    value = float(pd.Series(series).dropna().median())
    return value if np.isfinite(value) else float(default_value)


def _normalize_line_current_limits(net, nominal_max_i_ka=0.40):
    if "max_i_ka" not in net.line.columns:
        net.line["max_i_ka"] = nominal_max_i_ka
    max_i = net.line["max_i_ka"].astype(float)
    invalid = (~np.isfinite(max_i)) | (max_i <= 0.0) | (max_i > 5.0)
    net.line.loc[invalid, "max_i_ka"] = float(nominal_max_i_ka)
    return net


def _ensure_ieee33_tie_lines(net):
    _ensure_line_columns(net)
    max_i_ka = _median_or_default(net.line["max_i_ka"] if "max_i_ka" in net.line.columns else None, 0.40)
    c_nf_per_km = _median_or_default(net.line["c_nf_per_km"] if "c_nf_per_km" in net.line.columns else None, 0.0)
    for tie in STANDARD_TIE_BRANCH_DATA:
        mask = _line_mask_for_endpoints(net, tie["from_bus"], tie["to_bus"])
        matching_indices = list(net.line.index[mask])
        if matching_indices:
            net.line.loc[matching_indices, "is_tie"] = True
            net.line.loc[matching_indices, "in_service"] = False
            continue
        line_idx = pp.create_line_from_parameters(
            net,
            from_bus=tie["from_bus"],
            to_bus=tie["to_bus"],
            length_km=1.0,
            r_ohm_per_km=tie["r_ohm"],
            x_ohm_per_km=tie["x_ohm"],
            c_nf_per_km=c_nf_per_km,
            max_i_ka=max_i_ka,
            name=f"tie_{tie['from_bus'] + 1}_{tie['to_bus'] + 1}",
            in_service=False,
        )
        net.line.at[line_idx, "is_tie"] = True
    net.line["in_service"] = net.line["in_service"].astype(bool)
    net.line["is_tie"] = net.line["is_tie"].astype(bool)
    return net


def get_root_bus(net):
    if len(net.ext_grid) > 0 and "bus" in net.ext_grid.columns:
        return int(net.ext_grid.iloc[0]["bus"])
    return int(net.bus.index.min())


def load_ieee33():
    net = pn.case33bw()
    _ensure_ieee33_tie_lines(net)
    _normalize_line_current_limits(net, nominal_max_i_ka=0.40)
    base_graph = build_graph(net, active_only=True)
    if not validate_configuration(net):
        raise ValueError("Prepared IEEE-33 base configuration is not radial and connected.")
    return net, base_graph


## STEP 2 — Graph Builder and STEP 3 — Topology Metrics

The graph builder converts the pandapower network to NetworkX. Topology metrics use BFS and impedance-weighted shortest paths from the substation.


In [ ]:
def _bus_load_table(net):
    loads = pd.DataFrame(index=net.bus.index, data={"p_load": 0.0, "q_load": 0.0})
    if len(net.load) == 0:
        return loads
    load_df = net.load.copy()
    if "in_service" in load_df.columns:
        load_df = load_df[load_df["in_service"].fillna(True).astype(bool)]
    if len(load_df) == 0:
        return loads
    grouped = load_df.groupby("bus")[["p_mw", "q_mvar"]].sum()
    loads.loc[grouped.index, "p_load"] = grouped["p_mw"].astype(float)
    loads.loc[grouped.index, "q_load"] = grouped["q_mvar"].astype(float)
    return loads


def _initial_voltage_pu_table(net):
    voltage = pd.Series(1.0, index=net.bus.index, dtype=float)
    if len(net.ext_grid) > 0:
        for _, row in net.ext_grid.iterrows():
            vm_pu = float(row["vm_pu"]) if "vm_pu" in net.ext_grid.columns else 1.0
            voltage.loc[int(row["bus"])] = vm_pu
    return voltage


def line_is_closed(net, line_idx):
    closed = bool(net.line.at[line_idx, "in_service"])
    if "switch" in net and len(net.switch) > 0:
        switches = net.switch[(net.switch["et"] == "l") & (net.switch["element"].astype(int) == int(line_idx))]
        if len(switches) > 0:
            closed = closed and bool(switches["closed"].fillna(True).astype(bool).all())
    return bool(closed)


def _line_impedance(net, line_idx):
    row = net.line.loc[line_idx]
    length = float(row["length_km"]) if "length_km" in net.line.columns else 1.0
    r = float(row["r_ohm_per_km"]) * length if "r_ohm_per_km" in net.line.columns else 0.0
    x = float(row["x_ohm_per_km"]) * length if "x_ohm_per_km" in net.line.columns else 0.0
    return r, x


def build_graph(net, active_only=False):
    _ensure_line_columns(net)
    G = nx.Graph()
    load_table = _bus_load_table(net)
    voltage_table = _initial_voltage_pu_table(net)
    root_bus = get_root_bus(net)
    for bus_idx in net.bus.index:
        bus_idx_int = int(bus_idx)
        G.add_node(
            bus_idx_int,
            p_load=float(load_table.loc[bus_idx, "p_load"]),
            q_load=float(load_table.loc[bus_idx, "q_load"]),
            bus_type=1.0 if bus_idx_int == root_bus else 0.0,
            voltage=float(voltage_table.loc[bus_idx]),
        )
    for line_idx, row in net.line.iterrows():
        switch_status = 1.0 if line_is_closed(net, line_idx) else 0.0
        if active_only and switch_status < 0.5:
            continue
        r, x = _line_impedance(net, line_idx)
        G.add_edge(
            int(row["from_bus"]),
            int(row["to_bus"]),
            line_idx=int(line_idx),
            r=float(r),
            x=float(x),
            impedance=float(np.sqrt(r * r + x * x)),
            switch_status=float(switch_status),
            is_tie=bool(row["is_tie"]),
        )
    return G


def compute_feeder_depth(G, root_bus):
    depth = {int(node): -1 for node in G.nodes}
    if root_bus not in G.nodes:
        return depth
    for node, value in nx.single_source_shortest_path_length(G, int(root_bus)).items():
        depth[int(node)] = int(value)
    return depth


def compute_distance_to_substation(G, root_bus):
    distance = {int(node): np.inf for node in G.nodes}
    if root_bus not in G.nodes:
        return distance
    try:
        lengths = nx.single_source_dijkstra_path_length(G, int(root_bus), weight="impedance")
    except Exception:
        lengths = nx.single_source_shortest_path_length(G, int(root_bus))
    for node, value in lengths.items():
        distance[int(node)] = float(value)
    return distance


## STEP 4 — Physics-Calibrated Load Sampler

`generate_realistic_loads(seed)` uses bounded clipped normal sampling. It never samples arbitrary load multipliers outside the calibrated operating-regime bounds.


In [ ]:
def check_connectivity(G):
    return len(G.nodes) > 0 and nx.is_connected(G)


def check_radiality(G):
    return len(G.nodes) > 0 and nx.is_tree(G)


def validate_configuration(net):
    active_graph = build_graph(net, active_only=True)
    return check_connectivity(active_graph) and check_radiality(active_graph)


def prevalidate_before_powerflow(net):
    active_graph = build_graph(net, active_only=True)
    connected = check_connectivity(active_graph)
    radial = check_radiality(active_graph)
    return {
        "valid": bool(connected and radial),
        "connected": bool(connected),
        "radial": bool(radial),
        "nodes": int(active_graph.number_of_nodes()),
        "active_edges": int(active_graph.number_of_edges()),
    }


def _bounded_normal(rng, mean, std, low, high, size):
    return np.clip(rng.normal(loc=mean, scale=std, size=size), low, high)


def _enforce_total_load_bounds(scales, load_buses, base_bus_load, total_bounds, clip_bounds):
    low_total, high_total = total_bounds
    low_clip, high_clip = clip_bounds
    weights = np.array([base_bus_load.get(int(bus), 0.0) for bus in load_buses], dtype=float)
    if weights.sum() <= 0:
        return np.clip(scales, low_clip, high_clip)
    weighted_average = float(np.sum(scales * weights) / np.sum(weights))
    adjusted = scales.copy()
    if weighted_average > high_total:
        adjusted *= high_total / weighted_average
    elif weighted_average < low_total:
        adjusted *= low_total / max(weighted_average, 1e-9)
    return np.clip(adjusted, low_clip, high_clip)


def generate_realistic_loads(net, seed, requested_regime="normal"):
    rng = np.random.default_rng(int(seed))
    if requested_regime not in LOAD_SPECS:
        raise ValueError(f"Unknown requested_regime={requested_regime}")
    spec = LOAD_SPECS[requested_regime]
    scales = pd.Series(1.0, index=net.bus.index, dtype=float)
    load_table_before = _bus_load_table(net)
    base_bus_load = {int(idx): float(load_table_before.loc[idx, "p_load"]) for idx in load_table_before.index}
    if len(net.load) == 0:
        return scales.to_numpy(dtype=np.float32), {"requested_regime": requested_regime, "total_load_ratio": 1.0}
    load_buses = sorted(pd.unique(net.load["bus"].astype(int)))
    raw_scales = _bounded_normal(
        rng,
        mean=spec["mean"],
        std=spec["std"],
        low=spec["clip"][0],
        high=spec["clip"][1],
        size=len(load_buses),
    )
    bus_scales = _enforce_total_load_bounds(raw_scales, load_buses, base_bus_load, spec["total_bounds"], spec["clip"])
    for bus_idx, scale in zip(load_buses, bus_scales):
        scales.loc[int(bus_idx)] = float(scale)
    for load_idx, row in net.load.iterrows():
        scale = float(scales.loc[int(row["bus"])])
        net.load.at[load_idx, "p_mw"] = float(row["p_mw"]) * scale
        net.load.at[load_idx, "q_mvar"] = float(row["q_mvar"]) * scale
    load_after = _bus_load_table(net)
    base_total = max(float(load_table_before["p_load"].sum()), 1e-9)
    total_ratio = float(load_after["p_load"].sum() / base_total)
    return scales.to_numpy(dtype=np.float32), {
        "requested_regime": requested_regime,
        "distribution": "bounded_clipped_normal",
        "total_load_ratio": total_ratio,
        "scale_bounds": list(spec["clip"]),
        "total_bounds": list(spec["total_bounds"]),
    }


## STEP 5 — Restricted Switch Sampler and Base Topology Set

The notebook precomputes a small realistic domain of valid radial switch configurations (`30-80` total). Samples choose from this precomputed set and vary loads over those topologies. This follows the domain insight that IEEE-33 has limited realistic switching states.


In [ ]:
def _line_status_vector(net):
    return np.array([1 if line_is_closed(net, idx) else 0 for idx in net.line.index], dtype=np.int8)


def _apply_line_status(net, line_status):
    for line_idx, status in zip(net.line.index, np.asarray(line_status, dtype=np.int8)):
        net.line.at[line_idx, "in_service"] = bool(status)
    return net


def topology_signature_from_status(net, line_status):
    edges = []
    for status, (_, row) in zip(np.asarray(line_status, dtype=np.int8), net.line.iterrows()):
        if int(status) == 1:
            edges.append(tuple(sorted((int(row["from_bus"]), int(row["to_bus"])))) )
    return str(abs(hash(tuple(sorted(edges)))))


def switch_config_key_from_status(line_status):
    return ",".join(str(i) for i, status in enumerate(np.asarray(line_status, dtype=np.int8)) if int(status) == 1)


def _cycle_edges_from_nodes(net, cycle_nodes):
    cycle_set = set(int(node) for node in cycle_nodes)
    cycle_edges = []
    for line_idx, row in net.line.iterrows():
        if not line_is_closed(net, line_idx):
            continue
        if int(row["from_bus"]) in cycle_set and int(row["to_bus"]) in cycle_set:
            cycle_edges.append(int(line_idx))
    return cycle_edges


def _apply_branch_exchange_step(net, rng, base_depth):
    open_lines = [int(idx) for idx in net.line.index if not line_is_closed(net, idx)]
    if len(open_lines) == 0:
        raise ValueError("No open branches are available for branch exchange.")
    close_line = int(rng.choice(open_lines))
    net.line.at[close_line, "in_service"] = True
    trial_graph = build_graph(net, active_only=True)
    cycles = nx.cycle_basis(trial_graph)
    if len(cycles) == 0:
        net.line.at[close_line, "in_service"] = False
        raise ValueError("Closing branch did not create a cycle.")
    close_row = net.line.loc[close_line]
    close_endpoints = {int(close_row["from_bus"]), int(close_row["to_bus"])}
    selected_cycle = cycles[0]
    for cycle in cycles:
        if close_endpoints.issubset(set(int(node) for node in cycle)):
            selected_cycle = cycle
            break
    cycle_edges = _cycle_edges_from_nodes(net, selected_cycle)
    candidates = [idx for idx in cycle_edges if idx != close_line]
    if len(candidates) == 0:
        net.line.at[close_line, "in_service"] = False
        raise ValueError("No branch can be opened.")
    candidate_scores = []
    for line_idx in candidates:
        row = net.line.loc[line_idx]
        endpoint_depths = [base_depth.get(int(row["from_bus"]), 0), base_depth.get(int(row["to_bus"]), 0)]
        candidate_scores.append(max(endpoint_depths) + 1.0)
    inverse_scores = 1.0 / np.asarray(candidate_scores, dtype=float)
    probabilities = inverse_scores / inverse_scores.sum()
    open_line = int(rng.choice(candidates, p=probabilities))
    net.line.at[open_line, "in_service"] = False
    if not validate_configuration(net):
        net.line.at[open_line, "in_service"] = True
        net.line.at[close_line, "in_service"] = False
        raise ValueError("Branch exchange produced invalid topology.")
    open_row = net.line.loc[open_line]
    return {
        "closed_line": int(close_line),
        "closed_edge": [int(close_row["from_bus"]), int(close_row["to_bus"])],
        "opened_line": int(open_line),
        "opened_edge": [int(open_row["from_bus"]), int(open_row["to_bus"])],
        "cycle_length": int(len(selected_cycle)),
    }


def topology_base_case_screen_passes(net):
    """Reject radial topologies that are already weak under base loading."""
    pf = run_powerflow(net)
    if not pf.get("success", False):
        return False, pf.get("reason", "base_topology_powerflow_failed")
    if float(pf["min_voltage_pu"]) < 0.86:
        return False, "base_topology_voltage_screen"
    if float(pf["max_line_loading_percent"]) > 90.0:
        return False, "base_topology_loading_screen"
    if float(pf["total_loss_kw"]) > 500.0:
        return False, "base_topology_loss_screen"
    return True, "accepted"


def precompute_valid_topologies(base_net, target_count=TARGET_PRECOMPUTED_TOPOLOGIES, seed=GLOBAL_SEED):
    rng = np.random.default_rng(int(seed))
    topologies = []
    seen = set()
    base_status = _line_status_vector(base_net)
    base_signature = topology_signature_from_status(base_net, base_status)
    topologies.append({
        "topology_id": base_signature,
        "switch_config_key": switch_config_key_from_status(base_status),
        "line_status": base_status.astype(np.int8).tolist(),
        "switch_operations": [],
        "topology_deviation": 0,
        "num_branch_exchanges": 0,
    })
    seen.add(base_signature)
    base_graph = build_graph(base_net, active_only=True)
    base_depth = compute_feeder_depth(base_graph, get_root_bus(base_net))
    attempts = 0
    max_attempts = int(target_count * 120)
    while len(topologies) < min(target_count, MAX_PRECOMPUTED_TOPOLOGIES) and attempts < max_attempts:
        attempts += 1
        net = pp.from_json_string(pp.to_json(base_net))
        actions = []
        try:
            exchanges = int(rng.choice([1, 2], p=[0.75, 0.25]))
            for _ in range(exchanges):
                actions.append(_apply_branch_exchange_step(net, rng, base_depth))
            status = _line_status_vector(net)
            deviation = int(np.abs(status - base_status).sum())
            if deviation > MAX_SWITCH_DEVIATION:
                continue
            signature = topology_signature_from_status(net, status)
            if signature in seen:
                continue
            if not validate_configuration(net):
                continue
            screen_ok, _ = topology_base_case_screen_passes(net)
            if not screen_ok:
                continue
            topologies.append({
                "topology_id": signature,
                "switch_config_key": switch_config_key_from_status(status),
                "line_status": status.astype(np.int8).tolist(),
                "switch_operations": actions,
                "topology_deviation": deviation,
                "num_branch_exchanges": len(actions),
            })
            seen.add(signature)
        except Exception:
            continue
    if len(topologies) < MIN_PRECOMPUTED_TOPOLOGIES:
        raise RuntimeError(f"Only precomputed {len(topologies)} valid topologies; need at least {MIN_PRECOMPUTED_TOPOLOGIES}.")
    print(f"Precomputed {len(topologies)} valid radial topologies | attempts={attempts} | max_deviation={MAX_SWITCH_DEVIATION}")
    return topologies


def sample_switch_configuration(valid_topologies, seed):
    rng = np.random.default_rng(int(seed))
    index = int(rng.integers(0, len(valid_topologies)))
    return valid_topologies[index]



## STEP 6 — Pre-Flow Filter, STEP 7 — Power Flow, and STEP 8 — Feature Extraction

The early filter estimates candidate risk before `pp.runpp`. The hard constraint gate after power flow strictly rejects voltage collapse, overloads, unrealistic losses, and invalid topologies.


In [ ]:
def estimate_preflow_risk(net, load_metadata, topology_record):
    active_graph = build_graph(net, active_only=True)
    root_bus = get_root_bus(net)
    depth = compute_feeder_depth(active_graph, root_bus)
    load_table = _bus_load_table(net)
    depths = np.array([max(depth.get(int(bus), 0), 0) for bus in load_table.index], dtype=float)
    loads = load_table["p_load"].to_numpy(dtype=float)
    if loads.sum() > 0 and depths.max() > 0:
        depth_imbalance = float(np.sum(loads * depths) / (loads.sum() * depths.max()))
    else:
        depth_imbalance = 0.0
    load_ratio = float(load_metadata.get("total_load_ratio", 1.0))
    load_risk = float(np.clip((load_ratio - 0.95) / 0.35, 0.0, 1.0))
    topology_risk = float(np.clip(topology_record.get("topology_deviation", 0) / max(MAX_SWITCH_DEVIATION, 1), 0.0, 1.0))
    feeder_stress = 0.55 * load_risk + 0.30 * depth_imbalance + 0.15 * topology_risk
    return {
        "risk_score": float(feeder_stress),
        "load_risk": load_risk,
        "depth_imbalance": depth_imbalance,
        "topology_risk": topology_risk,
    }


def preflow_filter_passes(risk, requested_regime):
    limit = {"normal": 0.82, "mild_stress": 0.90, "extreme_stress": 0.96}[requested_regime]
    return bool(float(risk["risk_score"]) <= limit)


def _has_powerflow_initialization(net):
    return (
        hasattr(net, "res_bus")
        and len(net.res_bus) == len(net.bus)
        and "vm_pu" in net.res_bus.columns
        and np.isfinite(net.res_bus["vm_pu"].to_numpy(dtype=float)).all()
    )


def run_powerflow(net):
    topology_check = prevalidate_before_powerflow(net)
    if not topology_check["valid"]:
        return {"success": False, "reason": "pre_powerflow_invalid_topology", "topology_check": topology_check}
    init_mode = "results" if _has_powerflow_initialization(net) else "flat"
    try:
        pp.runpp(net, algorithm="nr", init=init_mode, calculate_voltage_angles=False, enforce_q_lims=False, max_iteration=50, tolerance_mva=1e-8, numba=False)
    except Exception as first_exc:
        if init_mode == "results":
            try:
                pp.runpp(net, algorithm="nr", init="flat", calculate_voltage_angles=False, enforce_q_lims=False, max_iteration=50, tolerance_mva=1e-8, numba=False)
            except Exception as second_exc:
                return {"success": False, "reason": f"powerflow_failed: {type(second_exc).__name__}: {second_exc}"}
        else:
            return {"success": False, "reason": f"powerflow_failed: {type(first_exc).__name__}: {first_exc}"}
    if not bool(getattr(net, "converged", False)) and init_mode == "results":
        try:
            pp.runpp(net, algorithm="nr", init="flat", calculate_voltage_angles=False, enforce_q_lims=False, max_iteration=50, tolerance_mva=1e-8, numba=False)
        except Exception as exc:
            return {"success": False, "reason": f"powerflow_failed: {type(exc).__name__}: {exc}"}
    if not bool(getattr(net, "converged", False)):
        return {"success": False, "reason": "powerflow_not_converged"}
    total_loss_kw = 0.0
    if "res_line" in net and len(net.res_line) > 0 and "pl_mw" in net.res_line.columns:
        total_loss_kw += float(np.nan_to_num(net.res_line["pl_mw"].to_numpy(dtype=float), nan=0.0).sum() * 1000.0)
    vm_pu = np.nan_to_num(net.res_bus["vm_pu"].to_numpy(dtype=float), nan=np.nan)
    loading = np.array([0.0], dtype=float)
    if "res_line" in net and len(net.res_line) > 0 and "loading_percent" in net.res_line.columns:
        loading = np.nan_to_num(net.res_line["loading_percent"].to_numpy(dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    voltage_violation_count = int(((vm_pu < VOLTAGE_NORMAL_MIN) | (vm_pu > VOLTAGE_MAX)).sum())
    overload_count = int((loading > LOADING_HARD_MAX).sum())
    return {
        "success": True,
        "total_loss_kw": float(total_loss_kw),
        "min_voltage_pu": float(np.nanmin(vm_pu)),
        "max_line_loading_percent": float(np.nanmax(loading)),
        "voltage_violation_count": voltage_violation_count,
        "overload_count": overload_count,
        "constraint_violation_count": int(voltage_violation_count + overload_count),
    }


def classify_operating_category(pf_result):
    min_v = float(pf_result["min_voltage_pu"])
    loading = float(pf_result["max_line_loading_percent"])
    loss = float(pf_result["total_loss_kw"])
    if min_v < VOLTAGE_MILD_MIN or loading >= LOADING_MILD_MAX or loss >= LOSS_MILD_MAX_KW:
        return "extreme_stress"
    if min_v < VOLTAGE_NORMAL_MIN or loading >= LOADING_NORMAL_MAX or loss >= LOSS_NORMAL_MAX_KW:
        return "mild_stress"
    return "normal"


def hard_constraint_gate(net, pf_result):
    if not pf_result.get("success", False):
        return False, pf_result.get("reason", "powerflow_failed")
    if not validate_configuration(net):
        return False, "invalid_topology_post_powerflow"
    min_v = float(pf_result["min_voltage_pu"])
    loading = float(pf_result["max_line_loading_percent"])
    loss = float(pf_result["total_loss_kw"])
    if min_v < VOLTAGE_HARD_MIN:
        return False, "hard_voltage_reject"
    if min_v > VOLTAGE_MAX:
        return False, "high_voltage_reject"
    if loading > LOADING_HARD_MAX:
        return False, "hard_loading_reject"
    if loss > LOSS_HARD_MAX_KW:
        return False, "hard_loss_reject"
    if loss < MIN_REALISTIC_LOSS_KW:
        return False, "low_loss_outlier_reject"
    return True, "accepted"


def initialize_base_network():
    global BASE_NET_JSON, BASE_LINE_STATUS, BASE_GRAPH, BASE_TOTAL_LOAD_MW, BASE_POWERFLOW_RESULT, VALID_TOPOLOGIES
    base_net, BASE_GRAPH = load_ieee33()
    BASE_LINE_STATUS = _line_status_vector(base_net)
    BASE_TOTAL_LOAD_MW = float(_bus_load_table(base_net)["p_load"].sum())
    BASE_POWERFLOW_RESULT = run_powerflow(base_net)
    if not BASE_POWERFLOW_RESULT.get("success", False):
        raise RuntimeError(f"Base IEEE-33 power flow failed: {BASE_POWERFLOW_RESULT}")
    BASE_NET_JSON = pp.to_json(base_net)
    VALID_TOPOLOGIES = precompute_valid_topologies(base_net, TARGET_PRECOMPUTED_TOPOLOGIES, GLOBAL_SEED)
    print(
        "Base IEEE-33 cached | "
        f"buses={len(base_net.bus)} | lines={len(base_net.line)} | base_load={BASE_TOTAL_LOAD_MW:.3f} MW | "
        f"base_min_v={BASE_POWERFLOW_RESULT['min_voltage_pu']:.4f} | base_max_loading={BASE_POWERFLOW_RESULT['max_line_loading_percent']:.2f}% | "
        f"valid_topologies={len(VALID_TOPOLOGIES)}"
    )
    return base_net, BASE_GRAPH


def clone_base_network():
    if BASE_NET_JSON is None:
        initialize_base_network()
    return pp.from_json_string(BASE_NET_JSON)


def extract_graph_features(net, topology_record, G=None):
    if G is None:
        G = build_graph(net, active_only=False)
    active_graph = build_graph(net, active_only=True)
    root_bus = get_root_bus(net)
    depth = compute_feeder_depth(active_graph, root_bus)
    distance = compute_distance_to_substation(active_graph, root_bus)
    load_table = _bus_load_table(net)
    voltage_table = _initial_voltage_pu_table(net)
    node_features = []
    for bus_idx in net.bus.index:
        bus_idx_int = int(bus_idx)
        node_features.append([
            float(load_table.loc[bus_idx, "p_load"]),
            float(load_table.loc[bus_idx, "q_load"]),
            float(depth.get(bus_idx_int, -1)),
            float(distance.get(bus_idx_int, np.inf)),
            1.0 if bus_idx_int == root_bus else 0.0,
            float(voltage_table.loc[bus_idx]),
        ])
    edge_features = []
    for line_idx, _ in net.line.iterrows():
        r, x = _line_impedance(net, line_idx)
        edge_features.append([float(r), float(x), 1.0 if line_is_closed(net, line_idx) else 0.0, float(np.sqrt(r * r + x * x))])
    global_features = np.array([
        float(load_table["p_load"].sum()),
        float(len(net.bus)),
        float(len(net.line)),
        float(sum(1 for idx in net.line.index if line_is_closed(net, idx))),
        float(topology_record.get("topology_deviation", 0)),
    ], dtype=np.float32)
    return {
        "node_features": np.asarray(node_features, dtype=np.float32),
        "edge_features": np.asarray(edge_features, dtype=np.float32),
        "global_features": global_features,
    }


## STEP 9 — HDF5 Streaming Storage and STEP 10/11 — Constraint-First Sample Generation

Workers generate candidate samples, but HDF5 writes are performed by a single writer in the notebook process. Full graph tensors are never accumulated for the whole dataset.


In [ ]:
def initialize_hdf5(path, overwrite=True):
    mode = "w" if overwrite else "a"
    with h5py.File(path, mode, track_order=True) as h5:
        h5.attrs["schema_version"] = "realistic_constraint_first_flat_v1"
        h5.attrs["dataset_name"] = "ieee33_realistic_graph_transformer_dataset"
        h5.attrs["global_seed"] = int(GLOBAL_SEED)
        h5.attrs["node_feature_names"] = json.dumps(NODE_FEATURE_NAMES)
        h5.attrs["edge_feature_names"] = json.dumps(EDGE_FEATURE_NAMES)
        h5.attrs["global_feature_names"] = json.dumps(GLOBAL_FEATURE_NAMES)
        h5.attrs["label_names"] = json.dumps(LABEL_NAMES)
        h5.attrs["target_category_policy"] = json.dumps(TARGET_CATEGORY_POLICY)
        h5.attrs["load_specs"] = json.dumps(LOAD_SPECS)
        h5.attrs["hard_constraints"] = json.dumps({
            "min_voltage_pu": VOLTAGE_HARD_MIN,
            "max_voltage_pu": VOLTAGE_MAX,
            "max_line_loading_percent": LOADING_HARD_MAX,
            "max_loss_kw": LOSS_HARD_MAX_KW,
            "max_switch_deviation": MAX_SWITCH_DEVIATION,
        })
        h5.attrs["total_attempted"] = 0
        h5.attrs["valid_samples"] = 0
        h5.attrs["rejected_samples"] = 0
        h5.attrs["powerflow_calls"] = 0
        h5.attrs["rejection_reasons"] = json.dumps({})
    return path


def _write_array(group, name, value):
    group.create_dataset(name, data=np.asarray(value), compression=COMPRESSION, compression_opts=COMPRESSION_OPTS, shuffle=True)


def write_records_to_hdf5(path, records, start_sample_id):
    string_dtype = h5py.string_dtype(encoding="utf-8")
    next_id = int(start_sample_id)
    with h5py.File(path, "a", track_order=True) as h5:
        for record in records:
            group = h5.create_group(f"sample_{next_id:06d}")
            _write_array(group, "node_features", record["node_features"])
            _write_array(group, "edge_features", record["edge_features"])
            _write_array(group, "global_features", record["global_features"])
            _write_array(group, "labels", record["labels"])
            group.create_dataset("metadata", data=json.dumps(record["metadata"]), dtype=string_dtype)
            group.attrs["seed"] = int(record["seed"])
            group.attrs["operating_category"] = record["metadata"]["operating_category"]
            group.attrs["requested_regime"] = record["metadata"]["requested_regime"]
            group.attrs["topology_id"] = record["metadata"]["topology_id"]
            group.attrs["topology_index"] = int(record["metadata"]["topology_index"])
            group.attrs["total_load_mw"] = float(record["global_features"][0])
            group.attrs["min_voltage_pu"] = float(record["labels"][1])
            group.attrs["max_line_loading_percent"] = float(record["labels"][2])
            group.attrs["loss_kw"] = float(record["labels"][0])
            next_id += 1
    return next_id


def update_hdf5_report(path, state):
    with h5py.File(path, "a") as h5:
        h5.attrs["total_attempted"] = int(state["total_attempted"])
        h5.attrs["valid_samples"] = int(state["valid_samples"])
        h5.attrs["rejected_samples"] = int(state["rejected_samples"])
        h5.attrs["powerflow_calls"] = int(state["powerflow_calls"])
        h5.attrs["rejection_reasons"] = json.dumps(state["rejection_reasons"])
        h5.attrs["category_counts"] = json.dumps(state["category_counts"])
        h5.attrs["topology_counts"] = json.dumps(state["topology_counts"])
        h5.attrs["precomputed_topology_count"] = int(len(VALID_TOPOLOGIES) if VALID_TOPOLOGIES is not None else 0)
        h5.attrs["base_powerflow_cached"] = bool(BASE_POWERFLOW_RESULT is not None and BASE_POWERFLOW_RESULT.get("success", False))


def _target_counts(n_samples):
    counts = {name: int(round(weight * n_samples)) for name, weight in TARGET_CATEGORY_POLICY.items()}
    delta = n_samples - sum(counts.values())
    counts["normal"] += delta
    return counts


def _choose_requested_regime(state, n_samples, rng):
    targets = _target_counts(n_samples)
    deficits = {name: max(targets[name] - state["category_counts"].get(name, 0), 0) for name in targets}
    if sum(deficits.values()) <= 0:
        names = list(TARGET_CATEGORY_POLICY.keys())
        weights = np.array([TARGET_CATEGORY_POLICY[name] for name in names], dtype=float)
    else:
        names = list(deficits.keys())
        weights = np.array([deficits[name] for name in names], dtype=float)
    weights = weights / weights.sum()
    return str(rng.choice(names, p=weights))


def _category_has_capacity(state, category, n_samples):
    targets = _target_counts(n_samples)
    return int(state["category_counts"].get(category, 0)) < int(targets[category])


def _build_record(seed, net, load_scaling, load_metadata, topology_record, topology_index, risk, pf_result, category):
    features = extract_graph_features(net, topology_record, build_graph(net, active_only=False))
    labels = np.array([
        pf_result["total_loss_kw"],
        pf_result["min_voltage_pu"],
        pf_result["max_line_loading_percent"],
        pf_result["voltage_violation_count"],
        pf_result["overload_count"],
        pf_result["constraint_violation_count"],
    ], dtype=np.float32)
    metadata = {
        "seed": int(seed),
        "requested_regime": load_metadata["requested_regime"],
        "operating_category": category,
        "load_scaling_vector": [float(value) for value in load_scaling],
        "load_metadata": load_metadata,
        "topology_index": int(topology_index),
        "topology_id": topology_record["topology_id"],
        "switch_config_key": topology_record["switch_config_key"],
        "switch_operations": topology_record["switch_operations"],
        "topology_deviation": int(topology_record["topology_deviation"]),
        "num_branch_exchanges": int(topology_record["num_branch_exchanges"]),
        "preflow_risk": risk,
    }
    return {
        "success": True,
        "seed": int(seed),
        "node_features": features["node_features"],
        "edge_features": features["edge_features"],
        "global_features": features["global_features"],
        "labels": labels,
        "metadata": metadata,
        "powerflow_called": True,
    }


def generate_sample(seed, requested_regime, topology_record, topology_index):
    try:
        net = clone_base_network()
        _apply_line_status(net, topology_record["line_status"])
        if not validate_configuration(net):
            return {"success": False, "seed": int(seed), "reason": "invalid_precomputed_topology", "powerflow_called": False}
        load_scaling, load_metadata = generate_realistic_loads(net, int(seed) + 101, requested_regime)
        risk = estimate_preflow_risk(net, load_metadata, topology_record)
        if not preflow_filter_passes(risk, requested_regime):
            return {"success": False, "seed": int(seed), "reason": "early_preflow_risk_reject", "powerflow_called": False}
        pf_result = run_powerflow(net)
        gate_ok, gate_reason = hard_constraint_gate(net, pf_result)
        if not gate_ok:
            return {"success": False, "seed": int(seed), "reason": gate_reason, "powerflow_called": True}
        category = classify_operating_category(pf_result)
        return _build_record(seed, net, load_scaling, load_metadata, topology_record, topology_index, risk, pf_result, category)
    except Exception as exc:
        return {"success": False, "seed": int(seed), "reason": f"sample_failed: {type(exc).__name__}: {exc}", "powerflow_called": False}


def generate_dataset(n_samples=N_SAMPLES, h5_path=H5_PATH, batch_size=BATCH_SIZE, n_jobs=N_JOBS, overwrite=True):
    initialize_base_network()
    h5_path = initialize_hdf5(h5_path, overwrite=overwrite)
    rng = np.random.default_rng(GLOBAL_SEED)
    state = {
        "total_attempted": 0,
        "valid_samples": 0,
        "rejected_samples": 0,
        "powerflow_calls": 0,
        "rejection_reasons": {},
        "category_counts": {name: 0 for name in TARGET_CATEGORY_POLICY},
        "topology_counts": {},
    }
    next_seed = int(GLOBAL_SEED)
    next_sample_id = 1
    max_attempts = int(np.ceil(n_samples * MAX_GENERATION_ATTEMPT_MULTIPLIER))
    progress = tqdm(total=n_samples, desc="Accepted realistic IEEE-33 samples", unit="sample")
    while state["valid_samples"] < n_samples and state["total_attempted"] < max_attempts:
        remaining = n_samples - state["valid_samples"]
        candidate_count = min(batch_size, max(remaining * 2, 1))
        plans = []
        for _ in range(candidate_count):
            requested_regime = _choose_requested_regime(state, n_samples, rng)
            topology_index = int(rng.integers(0, len(VALID_TOPOLOGIES)))
            plans.append((next_seed, requested_regime, VALID_TOPOLOGIES[topology_index], topology_index))
            next_seed += 1
        results = Parallel(n_jobs=n_jobs, backend="loky")(
            delayed(generate_sample)(seed, requested_regime, topology_record, topology_index)
            for seed, requested_regime, topology_record, topology_index in plans
        )
        accepted_records = []
        for result in results:
            if state["valid_samples"] >= n_samples:
                break
            state["total_attempted"] += 1
            if result.get("powerflow_called", False):
                state["powerflow_calls"] += 1
            if result.get("success", False):
                category = result["metadata"]["operating_category"]
                if not _category_has_capacity(state, category, n_samples):
                    state["rejected_samples"] += 1
                    reason = "category_capacity_reject"
                    state["rejection_reasons"][reason] = int(state["rejection_reasons"].get(reason, 0)) + 1
                    continue
                topology_id = result["metadata"]["topology_id"]
                state["category_counts"][category] = int(state["category_counts"].get(category, 0)) + 1
                state["topology_counts"][topology_id] = int(state["topology_counts"].get(topology_id, 0)) + 1
                state["valid_samples"] += 1
                accepted_records.append(result)
            else:
                state["rejected_samples"] += 1
                reason = result.get("reason", "unknown")
                state["rejection_reasons"][reason] = int(state["rejection_reasons"].get(reason, 0)) + 1
        if accepted_records:
            next_sample_id = write_records_to_hdf5(h5_path, accepted_records, next_sample_id)
            progress.update(len(accepted_records))
        update_hdf5_report(h5_path, state)
    progress.close()
    if state["valid_samples"] < n_samples:
        raise RuntimeError(f"Only generated {state['valid_samples']} of {n_samples} samples. Rejections: {state['rejection_reasons']}")
    acceptance_rate = 100.0 * state["valid_samples"] / max(state["total_attempted"], 1)
    report = {
        "h5_path": h5_path,
        "total_attempted": int(state["total_attempted"]),
        "valid_samples": int(state["valid_samples"]),
        "rejected_samples": int(state["rejected_samples"]),
        "acceptance_rate_percent": float(acceptance_rate),
        "rejection_rate_percent": float(100.0 - acceptance_rate),
        "powerflow_calls": int(state["powerflow_calls"]),
        "powerflow_calls_per_valid_sample": float(state["powerflow_calls"] / max(state["valid_samples"], 1)),
        "category_counts": state["category_counts"],
        "precomputed_topology_count": int(len(VALID_TOPOLOGIES)),
        "used_topology_count": int(len(state["topology_counts"])),
        "rejection_reasons": state["rejection_reasons"],
    }
    print(json.dumps(report, indent=2))
    return report


# Run the production default dataset generation.
generation_report = generate_dataset(N_SAMPLES, H5_PATH, BATCH_SIZE, N_JOBS, overwrite=True)


## STEP 12 — Validation Metrics and Plotly Analysis

The analysis reads scalar labels and metadata only. It reports acceptance rate, topology reuse, voltage/loss/loading distributions, and constraint-failure breakdown.


In [ ]:
def _read_metadata_dataset(dataset):
    raw = dataset[()]
    if isinstance(raw, bytes):
        raw = raw.decode("utf-8")
    return json.loads(raw)


def collect_dataset_analysis(h5_path=H5_PATH):
    rows = []
    load_scales = []
    attrs = {}
    with h5py.File(h5_path, "r") as h5:
        attrs = {key: h5.attrs[key] for key in h5.attrs.keys()}
        sample_names = sorted(name for name in h5.keys() if name.startswith("sample_"))
        for sample_name in tqdm(sample_names, desc="Reading scalar summaries", unit="sample"):
            group = h5[sample_name]
            labels = group["labels"][:]
            global_features = group["global_features"][:]
            metadata = _read_metadata_dataset(group["metadata"])
            load_scales.extend(metadata["load_scaling_vector"])
            rows.append({
                "sample": sample_name,
                "seed": int(metadata["seed"]),
                "requested_regime": metadata["requested_regime"],
                "operating_category": metadata["operating_category"],
                "topology_id": metadata["topology_id"],
                "topology_index": int(metadata["topology_index"]),
                "topology_deviation": int(metadata["topology_deviation"]),
                "total_load_mw": float(global_features[0]),
                "total_loss_kw": float(labels[0]),
                "min_voltage_pu": float(labels[1]),
                "max_line_loading_percent": float(labels[2]),
                "voltage_violation_count": int(labels[3]),
                "overload_count": int(labels[4]),
                "constraint_violation_count": int(labels[5]),
                "risk_score": float(metadata["preflow_risk"]["risk_score"]),
                "load_ratio": float(metadata["load_metadata"]["total_load_ratio"]),
            })
    return pd.DataFrame(rows), pd.Series(load_scales, name="load_scaling_factor", dtype=float), attrs


def print_validation_report(df, attrs):
    total = int(attrs.get("total_attempted", 0))
    valid = int(attrs.get("valid_samples", len(df)))
    rejected = int(attrs.get("rejected_samples", 0))
    acceptance = 100.0 * valid / max(total, 1)
    topology_counts = df["topology_id"].value_counts()
    print("Dataset Validation Report")
    print("-" * 32)
    print(f"Total attempts        : {total:,}")
    print(f"Accepted samples      : {valid:,}")
    print(f"Rejected candidates   : {rejected:,}")
    print(f"Acceptance rate       : {acceptance:.2f}%")
    print(f"Precomputed topologies: {int(attrs.get('precomputed_topology_count', 0))}")
    print(f"Used topologies       : {len(topology_counts)}")
    print(f"Topology reuse min/med/max: {int(topology_counts.min())}/{float(topology_counts.median()):.1f}/{int(topology_counts.max())}")
    print(f"Operating categories  : {df['operating_category'].value_counts().to_dict()}")
    print(f"Voltage min/max       : {df['min_voltage_pu'].min():.4f} / {df['min_voltage_pu'].max():.4f}")
    print(f"Loading max           : {df['max_line_loading_percent'].max():.2f}%")
    print(f"Loss min/max          : {df['total_loss_kw'].min():.2f} / {df['total_loss_kw'].max():.2f} kW")
    print(f"Constraint failures   : {json.loads(attrs.get('rejection_reasons', '{}'))}")


def plot_voltage_distribution(df):
    fig = px.histogram(df, x="min_voltage_pu", color="operating_category", nbins=60, title="Voltage Distribution by Operating Category")
    fig.add_vline(x=VOLTAGE_NORMAL_MIN, line_dash="dash", line_color="green", annotation_text="0.90")
    fig.add_vline(x=VOLTAGE_MILD_MIN, line_dash="dash", line_color="orange", annotation_text="0.85")
    fig.add_vline(x=VOLTAGE_HARD_MIN, line_dash="dash", line_color="red", annotation_text="0.80 hard")
    fig.show()


def plot_loss_distribution(df):
    fig = px.histogram(df, x="total_loss_kw", color="operating_category", nbins=60, title="Loss Distribution")
    fig.add_vline(x=LOSS_NORMAL_MAX_KW, line_dash="dash", line_color="green", annotation_text="300 kW")
    fig.add_vline(x=LOSS_MILD_MAX_KW, line_dash="dash", line_color="orange", annotation_text="500 kW")
    fig.add_vline(x=LOSS_HARD_MAX_KW, line_dash="dash", line_color="red", annotation_text="600 kW hard")
    fig.show()


def plot_loading_distribution(df):
    fig = px.histogram(df, x="max_line_loading_percent", color="operating_category", nbins=60, title="Line Loading Distribution")
    fig.add_vline(x=LOADING_NORMAL_MAX, line_dash="dash", line_color="green", annotation_text="80%")
    fig.add_vline(x=LOADING_MILD_MAX, line_dash="dash", line_color="orange", annotation_text="95%")
    fig.add_vline(x=LOADING_HARD_MAX, line_dash="dash", line_color="red", annotation_text="100% hard")
    fig.show()


def plot_topology_reuse(df):
    counts = df["topology_id"].value_counts().reset_index()
    counts.columns = ["topology_id", "count"]
    fig = px.bar(counts.head(80), x="topology_id", y="count", title="Topology Reuse Distribution")
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()


def plot_load_scaling_distribution(load_scale_series):
    fig = px.histogram(load_scale_series.to_frame(), x="load_scaling_factor", nbins=70, title="Load Scaling Distribution")
    fig.show()


def plot_correlation_heatmap(df):
    columns = ["total_load_mw", "total_loss_kw", "min_voltage_pu", "max_line_loading_percent", "constraint_violation_count", "risk_score"]
    corr = df[columns].corr()
    fig = go.Figure(data=go.Heatmap(z=corr.values, x=corr.columns, y=corr.index, colorscale="RdBu", zmin=-1, zmax=1))
    fig.update_layout(title="Correlation Heatmap")
    fig.show()


def plot_constraint_violation_stats(df):
    fig = px.histogram(df, x="constraint_violation_count", color="operating_category", nbins=30, title="Constraint Violations in Accepted Samples")
    fig.show()


analysis_df, load_scale_series, dataset_attrs = collect_dataset_analysis(H5_PATH)
if len(analysis_df) == 0:
    raise RuntimeError("No valid samples were generated; analysis cannot continue.")
print_validation_report(analysis_df, dataset_attrs)
plot_voltage_distribution(analysis_df)
plot_loss_distribution(analysis_df)
plot_loading_distribution(analysis_df)
plot_topology_reuse(analysis_df)
plot_load_scaling_distribution(load_scale_series)
plot_correlation_heatmap(analysis_df)
plot_constraint_violation_stats(analysis_df)
analysis_df.describe(include="all")


## STEP 13 — Final Dataset Summary

The final summary reports sample counts, feature dimensions, label definitions, hard constraints, and the HDF5 flat storage schema.


In [ ]:
def print_dataset_summary(h5_path=H5_PATH):
    with h5py.File(h5_path, "r") as h5:
        sample_names = sorted(name for name in h5.keys() if name.startswith("sample_"))
        if len(sample_names) == 0:
            raise RuntimeError("HDF5 file contains no sample groups.")
        first = h5[sample_names[0]]
        total = int(h5.attrs.get("total_attempted", 0))
        valid = int(h5.attrs.get("valid_samples", len(sample_names)))
        rejected = int(h5.attrs.get("rejected_samples", 0))
        acceptance = 100.0 * valid / max(total, 1)
        print("Dataset Ready for Graph Transformer Training")
        print("=" * 52)
        print(f"HDF5 path              : {h5_path}")
        print(f"Total attempts         : {total:,}")
        print(f"Valid samples          : {valid:,}")
        print(f"Rejected candidates    : {rejected:,}")
        print(f"Acceptance rate        : {acceptance:.2f}%")
        print(f"Precomputed topologies : {int(h5.attrs.get('precomputed_topology_count', 0))}")
        print(f"Node feature shape     : {first['node_features'].shape}")
        print(f"Edge feature shape     : {first['edge_features'].shape}")
        print(f"Global feature shape   : {first['global_features'].shape}")
        print(f"Labels shape           : {first['labels'].shape}")
        print(f"Node features          : {json.loads(h5.attrs['node_feature_names'])}")
        print(f"Edge features          : {json.loads(h5.attrs['edge_feature_names'])}")
        print(f"Global features        : {json.loads(h5.attrs['global_feature_names'])}")
        print(f"Labels                 : {json.loads(h5.attrs['label_names'])}")
        print("")
        print("Hard constraints")
        print(f"- min_voltage_pu >= {VOLTAGE_HARD_MIN}")
        print(f"- max_line_loading_percent <= {LOADING_HARD_MAX}")
        print(f"- total_loss_kw <= {LOSS_HARD_MAX_KW}")
        print(f"- topology must be connected and radial")
        print(f"- topology line-status deviation <= {MAX_SWITCH_DEVIATION}")
        print("")
        print("HDF5 storage schema")
        print("/sample_000001/")
        print("    node_features    float32 [num_buses, num_node_features]")
        print("    edge_features    float32 [num_lines, num_edge_features]")
        print("    global_features  float32 [num_global_features]")
        print("    labels           float32 [6]")
        print("    metadata         JSON string")


print_dataset_summary(H5_PATH)
